# NLP Project 5 — Sentiment Analysis Shopee Reviews Vietnamese

## Mục tiêu project

Trong project này, mình sẽ xây một hệ thống **Sentiment Analysis** cho review sản phẩm tiếng Việt.

Ví dụ:

```text
Input : Sản phẩm này rất tốt, giao hàng nhanh, đóng gói cẩn thận
Output: Positive
```

Project này dùng pipeline Machine Learning truyền thống:

```text
Vietnamese review
→ clean text
→ optional Vietnamese word segmentation
→ TF-IDF vectorization
→ SVM classifier
→ evaluate
→ error analysis
→ save model
→ predict review mới
```

Model chính:

```text
TF-IDF + Linear SVM
```

Dataset gợi ý:

- Kaggle: **Shopee Vietnamese Product Reviews Sentiment**
- Link tìm dataset: https://www.kaggle.com/datasets/dduongdev/shopee-vietnamese-product-reviews-sentiment

> Ghi chú quan trọng: notebook này không tự chứa dataset. Bạn tải dataset từ Kaggle về, giải nén vào thư mục `data/raw/`, sau đó chạy notebook từ trên xuống dưới.

# 0. Project overview

## Bài toán này là gì?

**Sentiment Analysis** là bài toán phân loại cảm xúc/ý kiến trong văn bản. Với review sản phẩm, model sẽ đọc câu review và dự đoán cảm xúc của khách hàng.

Ví dụ:

| Review | Label |
|---|---|
| `Sản phẩm đẹp, dùng rất ổn` | positive |
| `Giao hàng chậm, hàng bị lỗi` | negative |
| `Hàng bình thường, không có gì đặc biệt` | neutral |

Tùy dataset, nhãn có thể là:

- `positive`, `negative`
- hoặc `positive`, `neutral`, `negative`
- hoặc dạng số như `0`, `1`, `2`
- hoặc rating sao `1`, `2`, `3`, `4`, `5`

Notebook này viết linh hoạt để cố gắng tự nhận diện cột review và cột label.

# 1. Vì sao dùng TF-IDF + SVM?

## 1.1. Text không thể đưa trực tiếp vào model

Machine Learning model không hiểu câu chữ trực tiếp. Model cần dữ liệu dạng số.

Ví dụ câu:

```text
sản phẩm rất tốt
```

Cần biến thành vector số:

```text
[0.12, 0.00, 0.43, 0.88, ...]
```

Quá trình biến text thành vector gọi là **vectorization**.

## 1.2. TF-IDF là gì?

**TF-IDF** viết tắt của:

- **TF** = Term Frequency: một từ xuất hiện nhiều trong một document thì từ đó có thể quan trọng với document đó.
- **IDF** = Inverse Document Frequency: một từ xuất hiện quá nhiều trong mọi document thì từ đó ít đặc biệt hơn.

Công thức trực giác:

$$
TF\text{-}IDF(t, d) = TF(t, d) \times IDF(t)
$$

Trong đó:

$$
IDF(t) = \log\left(\frac{N}{DF(t)}\right)
$$

Ý nghĩa:

- Từ như `sản phẩm`, `shop`, `mình` có thể xuất hiện rất nhiều nên không phải lúc nào cũng phân biệt tốt.
- Từ như `tuyệt`, `ưng`, `lỗi`, `chậm`, `bể`, `thất vọng` thường hữu ích hơn cho sentiment.

## 1.3. SVM là gì?

**SVM - Support Vector Machine** là mô hình classification tìm một đường/ranh giới phân chia class sao cho margin lớn nhất.

Với text classification, sau TF-IDF, mỗi review là một vector rất dài và thưa (**high-dimensional sparse vector**). SVM thường hoạt động tốt với dạng dữ liệu này.

Trong project này ta dùng **LinearSVC**, tức SVM tuyến tính, vì:

- phù hợp với TF-IDF sparse matrix,
- chạy nhanh hơn SVC kernel RBF trên text lớn,
- dễ xem top từ quan trọng cho từng class thông qua hệ số `coef_`.

# 2. Cài đặt thư viện

Các thư viện chính:

- `pandas`: đọc và xử lý bảng dữ liệu
- `numpy`: tính toán số học
- `matplotlib`: vẽ biểu đồ
- `scikit-learn`: TF-IDF, SVM, train/test split, metrics, GridSearchCV
- `joblib`: lưu model

Nếu bạn muốn dùng tách từ tiếng Việt tốt hơn, có thể cài thêm `underthesea` hoặc `pyvi`. Notebook này sẽ chạy được dù chưa cài các thư viện tách từ.

In [ ]:
# Nếu chạy lần đầu và thiếu thư viện, bạn có thể mở comment để cài.
# !pip install pandas numpy matplotlib scikit-learn joblib
# Optional Vietnamese word segmentation:
# !pip install underthesea

In [51]:
import os
import re
import json
import unicodedata
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight

import joblib

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Import libraries thành công!")

Import libraries thành công!


# 3. Chuẩn bị dataset

## 3.1. Cấu trúc thư mục nên dùng

Bạn nên tạo folder project như sau:

```text
sentiment-analysis-shopee/
│
├── sentiment_analysis_shopee_tfidf_svm.ipynb
│
├── data/
│   └── raw/
│       └── <file_csv_download_tu_kaggle>.csv
│
├── models/
│   └── sentiment_tfidf_svm.joblib
│
└── reports/
    └── wrong_predictions.csv
```

## 3.2. Cách tải dataset từ Kaggle

Cách đơn giản nhất:

1. Vào link dataset Kaggle.
2. Bấm **Download**.
3. Giải nén file `.zip`.
4. Copy file `.csv` vào `data/raw/`.

Nếu dùng Kaggle API:

```bash
pip install kaggle
kaggle datasets download -d dduongdev/shopee-vietnamese-product-reviews-sentiment -p data/raw --unzip
```

In [52]:
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "shopee_reviews"
MODEL_DIR = PROJECT_DIR / "models" / "shopee_reviews"
REPORT_DIR = PROJECT_DIR / "models" / "shopee_reviews" / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("MODEL_DIR  :", MODEL_DIR)
print("REPORT_DIR :", REPORT_DIR)

PROJECT_DIR: c:\Users\ASUS\Documents\PROJECT\intro_to_AI
DATA_DIR   : c:\Users\ASUS\Documents\PROJECT\intro_to_AI\data\shopee_reviews
MODEL_DIR  : c:\Users\ASUS\Documents\PROJECT\intro_to_AI\models\shopee_reviews
REPORT_DIR : c:\Users\ASUS\Documents\PROJECT\intro_to_AI\models\shopee_reviews\reports


# 4. Load dữ liệu

Notebook này sẽ tìm tất cả file `.csv` trong `data/raw/`. Nếu có nhiều file, nó sẽ đọc file đầu tiên có vẻ đọc được.

Nếu dataset của bạn có tên file khác, không sao. Chỉ cần file đó nằm trong `data/raw/`.

In [53]:
import json

def read_data_with_fallback(path: Path) -> pd.DataFrame:
    """
    Đọc dữ liệu từ CSV hoặc JSON.

    Hỗ trợ:
    - .csv
    - .json dạng list of dict
    - .json dạng dict
    - .jsonl / newline-delimited JSON
    """

    suffix = path.suffix.lower()

    # =========================
    # Case 1: CSV file
    # =========================
    if suffix == ".csv":
        encodings = ["utf-8", "utf-8-sig", "latin1"]
        seps = [",", ";", "\t"]
        last_error = None

        for enc in encodings:
            for sep in seps:
                try:
                    df_try = pd.read_csv(path, encoding=enc, sep=sep)

                    # Nếu đọc ra >= 2 cột thì coi như đúng separator
                    if df_try.shape[1] >= 2:
                        print(
                            f"Đọc CSV thành công: {path.name} | "
                            f"encoding={enc} | sep={repr(sep)}"
                        )
                        return df_try

                except Exception as e:
                    last_error = e

        raise RuntimeError(f"Không đọc được CSV file {path}. Lỗi cuối: {last_error}")

    # =========================
    # Case 2: JSON file
    # =========================
    elif suffix == ".json":
        last_error = None

        # Cách 1: pandas đọc JSON thường
        try:
            df_try = pd.read_json(path, encoding="utf-8")
            print(f"Đọc JSON thành công bằng pd.read_json: {path.name}")
            return df_try
        except Exception as e:
            last_error = e

        # Cách 2: JSON lines
        try:
            df_try = pd.read_json(path, lines=True, encoding="utf-8")
            print(f"Đọc JSON Lines thành công: {path.name}")
            return df_try
        except Exception as e:
            last_error = e

        # Cách 3: đọc thủ công bằng json.load
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Nếu JSON là list of dict
            if isinstance(data, list):
                df_try = pd.DataFrame(data)
                print(f"Đọc JSON list thành công: {path.name}")
                return df_try

            # Nếu JSON là dict
            if isinstance(data, dict):
                # Trường hợp dict chứa key là danh sách reviews
                # Ví dụ: {"data": [...]} hoặc {"reviews": [...]}
                for key, value in data.items():
                    if isinstance(value, list):
                        df_try = pd.DataFrame(value)
                        print(f"Đọc JSON dict key='{key}' thành công: {path.name}")
                        return df_try

                # Nếu dict phẳng thì convert trực tiếp
                df_try = pd.DataFrame([data])
                print(f"Đọc JSON dict phẳng thành công: {path.name}")
                return df_try

        except Exception as e:
            last_error = e

        raise RuntimeError(f"Không đọc được JSON file {path}. Lỗi cuối: {last_error}")

    # =========================
    # Case 3: JSONL file
    # =========================
    elif suffix == ".jsonl":
        try:
            df_try = pd.read_json(path, lines=True, encoding="utf-8")
            print(f"Đọc JSONL thành công: {path.name}")
            return df_try
        except Exception as e:
            raise RuntimeError(f"Không đọc được JSONL file {path}. Lỗi: {e}")

    else:
        raise ValueError(
            f"File {path.name} không được hỗ trợ. "
            "Hiện chỉ hỗ trợ .csv, .json, .jsonl"
        )

data_files = sorted(
    list(DATA_DIR.glob("*.csv")) +
    list(DATA_DIR.glob("*.json")) +
    list(DATA_DIR.glob("*.jsonl"))
)

if len(data_files) == 0:
    raise FileNotFoundError(
        f"""
Không tìm thấy file dữ liệu trong thư mục: {DATA_DIR}

Bạn hãy tải dataset Shopee reviews từ Kaggle,
sau đó bỏ file .csv / .json / .jsonl vào thư mục này:

{DATA_DIR}
"""
    )

data_path = data_files[0]
print("File dữ liệu được chọn:", data_path)

df = read_data_with_fallback(data_path)

print("Shape:", df.shape)
df.head()

File dữ liệu được chọn: c:\Users\ASUS\Documents\PROJECT\intro_to_AI\data\shopee_reviews\aug_unaccented_reviews.jsonl
Đọc JSONL thành công: aug_unaccented_reviews.jsonl
Shape: (1348, 4)


,id,review,rating,label
0,59241444271,"Mui huong:Ngot nhe, thomm Phu hop voi loai da:minh nghi la moi loai da Cong dung:duong trang nhung ma ch thay trang chi thay thom nhe Helu mn, minh feeedback dayy. Minh nghi ai cung muon co loi k...",4,positive
1,75292434640,"Toi rat hai long voi dich vu cua nguoi ban va shipper lan nay. San pham toi dat mua khong chi chat luong tuyet voi ma con duoc giao dung hen, nhanh chong. Shipper rat nhiet tinh, chu dao va luon c...",5,positive
2,13364343551,"Review nhe mot so dong sua con uong tang can tot cho cm quan tam - Grow do: be uong de tieu hoa tuy nhien be nao uong hap thi len kg tot, k hap sua thi k cai thien nhieu - Dong Meiji la sua mat, s...",5,positive
3,84816190204,"Hieu qua:Sach Thiet ke:Dep Thiet ke nho gon, tien loi: Kich thuoc nho, trong luong nhe, de dang bo tui, balo, ly tuong cho viec di chuyen. Cao sach va nhanh: Luoi dao sac ben (thuong la thep khon...",5,positive
4,30102564756,"Giao hang nhanh, dong goi dep an toan Mui Pink khong ngot nhieu nhu review dau ma ngot mat, huong dau xit ra co mui nhu mui sua tam binh thuong, sau khi xit tam vai phut thi de lai mui thom hon, m...",4,positive


# 5. Khám phá cấu trúc dữ liệu ban đầu

Ở bước này, ta cần hiểu dataset có những cột nào.

Điều cần tìm:

- Cột chứa review text: thường có tên như `review`, `comment`, `content`, `text`, `sentence`.
- Cột chứa label/sentiment: thường có tên như `label`, `sentiment`, `rating`, `star`, `class`.

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nInfo:")
df.info()

In [ ]:
# Xem vài dòng ngẫu nhiên để hiểu dữ liệu.
pd.set_option("display.max_colwidth", 200)
df.sample(min(5, len(df)), random_state=RANDOM_STATE)

In [ ]:
# Kiểm tra missing values
missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (df.isna().mean() * 100).sort_values(ascending=False)

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_percent": missing_percent.round(2)
})
missing_df.head(20)

# 6. Tự động nhận diện cột text và label

Vì mỗi dataset Kaggle có thể đặt tên cột khác nhau, ta sẽ viết hàm tự đoán cột.

Nếu notebook đoán sai, bạn chỉ cần sửa thủ công hai biến:

```python
TEXT_COL = "tên_cột_review"
LABEL_COL = "tên_cột_label"
```

In [ ]:
def find_best_column(columns, candidates):
    # Tìm cột phù hợp nhất dựa trên danh sách tên ứng viên.
    cols_lower = {str(c).lower().strip(): c for c in columns}

    # Match chính xác
    for cand in candidates:
        cand_lower = cand.lower().strip()
        if cand_lower in cols_lower:
            return cols_lower[cand_lower]

    # Match chứa từ khóa
    for cand in candidates:
        cand_lower = cand.lower().strip()
        for col_lower, original_col in cols_lower.items():
            if cand_lower in col_lower:
                return original_col
    return None

text_candidates = [
    "review", "comment", "content", "text", "sentence", "feedback", 
    "body", "message", "description", "user_comment"
]

label_candidates = [
    "sentiment", "label", "class", "target", "polarity", "rating", 
    "star", "stars", "score", "rate"
]

TEXT_COL = find_best_column(df.columns, text_candidates)
LABEL_COL = find_best_column(df.columns, label_candidates)

print("TEXT_COL đoán được :", TEXT_COL)
print("LABEL_COL đoán được:", LABEL_COL)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        "Không tự nhận diện được TEXT_COL hoặc LABEL_COL. "
        "Hãy xem df.columns rồi gán thủ công TEXT_COL và LABEL_COL."
    )

In [ ]:
# Nếu notebook đoán sai, sửa thủ công tại đây rồi chạy lại các cell bên dưới.
# Ví dụ:
# TEXT_COL = "review"
# LABEL_COL = "sentiment"

print("Dùng cột text :", TEXT_COL)
print("Dùng cột label:", LABEL_COL)

data = df[[TEXT_COL, LABEL_COL]].copy()
data.columns = ["text", "label_raw"]

data.head()

# 7. Làm sạch label

Mục tiêu: đưa label về dạng dễ hiểu như:

```text
negative / neutral / positive
```

Vì dataset có thể dùng nhiều kiểu label khác nhau, ta xử lý các trường hợp phổ biến:

## Trường hợp A — label là string

Ví dụ:

```text
positive, negative, neutral
pos, neg, neu
Tích cực, Tiêu cực, Trung lập
```

## Trường hợp B — label là rating sao

Ví dụ:

```text
1, 2, 3, 4, 5
```

Ta map tạm như sau:

```text
1-2 sao → negative
3 sao   → neutral
4-5 sao → positive
```

Nếu dataset của bạn đã được gán nhãn sentiment sẵn, ta ưu tiên dùng nhãn đó thay vì rating.

In [ ]:
def normalize_label(x):
    # Chuẩn hóa label về negative / neutral / positive nếu có thể.
    if pd.isna(x):
        return np.nan

    # Nếu là số: có thể là rating hoặc label encoded.
    if isinstance(x, (int, float, np.integer, np.floating)):
        value = float(x)
        # Rating 1-5
        if value in [1, 2, 3, 4, 5]:
            if value <= 2:
                return "negative"
            elif value == 3:
                return "neutral"
            else:
                return "positive"
        # Binary encoded
        if value == 0:
            return "negative"
        if value == 1:
            return "positive"
        if value == 2:
            return "neutral"
        return str(x)

    s = str(x).strip().lower()
    s = unicodedata.normalize("NFC", s)

    # Một số mapping phổ biến
    mapping = {
        "positive": "positive", "pos": "positive", "p": "positive", "+": "positive",
        "tích cực": "positive", "tich cuc": "positive", "tich_cuc": "positive",
        "good": "positive", "happy": "positive", "like": "positive",
        "negative": "negative", "neg": "negative", "n": "negative", "-": "negative",
        "tiêu cực": "negative", "tieu cuc": "negative", "tieu_cuc": "negative",
        "bad": "negative", "unhappy": "negative", "dislike": "negative",
        "neutral": "neutral", "neu": "neutral", "trung lập": "neutral",
        "trung lap": "neutral", "trung_lap": "neutral", "normal": "neutral",
    }

    if s in mapping:
        return mapping[s]

    # Nếu là số nhưng đang ở dạng string
    try:
        value = float(s)
        if value in [1, 2, 3, 4, 5]:
            if value <= 2:
                return "negative"
            elif value == 3:
                return "neutral"
            else:
                return "positive"
        if value == 0:
            return "negative"
        if value == 1:
            return "positive"
        if value == 2:
            return "neutral"
    except ValueError:
        pass

    return s

# Drop missing ở text/label raw trước
data = data.dropna(subset=["text", "label_raw"]).copy()
data["label"] = data["label_raw"].apply(normalize_label)

print("Label raw unique:")
print(data["label_raw"].value_counts(dropna=False).head(20))

print("\nLabel normalized unique:")
print(data["label"].value_counts(dropna=False).head(20))

In [ ]:
# Chỉ giữ các label sentiment chính nếu có.
valid_labels = ["negative", "neutral", "positive"]

before = len(data)
data = data[data["label"].isin(valid_labels)].copy()
after = len(data)

print(f"Số dòng trước khi lọc label: {before}")
print(f"Số dòng sau khi lọc label : {after}")
print(f"Đã loại bỏ               : {before - after}")

if len(data) == 0:
    raise ValueError(
        "Sau khi normalize label thì không còn dòng nào. "
        "Có thể dataset của bạn dùng label khác. Hãy kiểm tra data['label_raw'].value_counts()."
    )

data["label"].value_counts()

# 8. EDA — Exploratory Data Analysis

Ta sẽ xem nhanh:

1. Dataset có bao nhiêu review.
2. Mỗi class có bao nhiêu mẫu.
3. Review dài/ngắn ra sao.
4. Có duplicate không.
5. Một vài ví dụ theo từng class.

EDA giúp mình hiểu dữ liệu trước khi train. Đây là thói quen rất quan trọng trong project ML thật.

In [ ]:
print("Số dòng:", len(data))
print("Số label:", data["label"].nunique())
print("Các label:", sorted(data["label"].unique()))

data.head()

In [ ]:
label_counts = data["label"].value_counts().sort_index()
label_percent = (data["label"].value_counts(normalize=True).sort_index() * 100).round(2)

label_summary = pd.DataFrame({
    "count": label_counts,
    "percent": label_percent
})
label_summary

In [ ]:
plt.figure(figsize=(7, 4))
label_counts.plot(kind="bar")
plt.title("Phân phối label sentiment")
plt.xlabel("Label")
plt.ylabel("Số lượng review")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Độ dài review theo số ký tự và số token tách bằng khoảng trắng
eda_data = data.copy()
eda_data["char_len"] = eda_data["text"].astype(str).apply(len)
eda_data["word_len"] = eda_data["text"].astype(str).apply(lambda x: len(x.split()))

eda_data[["char_len", "word_len"]].describe().T

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(eda_data["word_len"], bins=50)
plt.title("Phân phối độ dài review theo số token khoảng trắng")
plt.xlabel("Số token")
plt.ylabel("Số review")
plt.show()

In [ ]:
# Kiểm tra duplicate text
num_dup = eda_data.duplicated(subset=["text"]).sum()
print("Số review bị duplicate:", num_dup)
print("Tỉ lệ duplicate (%):", round(num_dup / len(eda_data) * 100, 2))

In [ ]:
# Xem ví dụ theo từng class
for label in sorted(data["label"].unique()):
    print("=" * 80)
    print("LABEL:", label)
    samples = data[data["label"] == label].sample(min(3, (data["label"] == label).sum()), random_state=RANDOM_STATE)
    for i, text in enumerate(samples["text"], start=1):
        print(f"\nVí dụ {i}:")
        print(text)

# 9. Text preprocessing tiếng Việt

Text từ review Shopee thường có nhiều nhiễu:

```text
Sản phẩm đẹppppppppp :))) giao hàng nhanhhhhh, shop rep okkkk
```

Các vấn đề thường gặp:

- viết hoa/viết thường lẫn lộn,
- emoji, icon, dấu câu,
- kéo dài ký tự: `đẹpppp`, `ngonnnnn`,
- teencode: `ko`, `k`, `kh`, `sp`, `shop`, `dc`,
- URL, HTML, số điện thoại, ký tự lạ.

Trong project này, ta sẽ làm sạch ở mức vừa đủ cho mô hình TF-IDF + SVM.

## Lưu ý quan trọng

Không nên làm sạch quá mạnh. Vì trong sentiment, một số ký tự và từ đặc biệt có ý nghĩa:

- `:))`, `😍`, `❤️` thường nghiêng positive.
- `:(`, `😡`, `huhu` thường nghiêng negative.
- Dấu `!` đôi khi thể hiện cảm xúc mạnh.

Notebook này sẽ map một số emoji/icon thành token như `emoji_positive`, `emoji_negative` thay vì xóa hết ngay từ đầu.

In [ ]:
# Stopwords tiếng Việt nhỏ gọn. Không nên quá aggressive trong sentiment analysis.
# Có thể mở rộng sau nếu muốn.
VI_STOPWORDS = set("và là của có cho với một những các này kia đó thì mà ở trong trên dưới rất quá hơi cũng đã đang sẽ được bị vào ra khi nếu vì nên như để từ tôi mình bạn anh chị em shop sản phẩm hàng".split())

TEENCODE_MAP = {
    "ko": "không",
    "k": "không",
    "kh": "không",
    "khg": "không",
    "hong": "không",
    "hok": "không",
    "kg": "không",
    "dc": "được",
    "đc": "được",
    "sp": "sản phẩm",
    "mn": "mọi người",
    "mng": "mọi người",
    "mik": "mình",
    "mk": "mình",
    "minh": "mình",
    "bt": "bình thường",
    "bth": "bình thường",
    "ok": "ổn",
    "oke": "ổn",
    "okie": "ổn",
    "tks": "cảm ơn",
    "thanks": "cảm ơn",
    "thank": "cảm ơn",
    "ship": "giao hàng",
    "shope": "shop",
}

POSITIVE_EMOJI_PATTERN = r"(😍|🥰|😘|❤️|❤|💕|💖|👍|😊|😁|😄|😆|😂|:D|:\)+|=\)+)"
NEGATIVE_EMOJI_PATTERN = r"(😡|😠|😭|😢|☹️|🙁|👎|:\(+|=\(+|haiz|huhu)"

def reduce_repeated_chars(text: str, max_repeat: int = 2) -> str:
    # Giảm ký tự lặp quá nhiều: đẹppppp -> đẹpp.
    pattern = r"(.)\1{" + str(max_repeat) + r",}"
    return re.sub(pattern, r"\1" * max_repeat, text)


def normalize_teencode(text: str) -> str:
    tokens = text.split()
    normalized_tokens = []
    for tok in tokens:
        normalized_tokens.append(TEENCODE_MAP.get(tok, tok))
    return " ".join(normalized_tokens)


def clean_text(text: str, remove_stopwords: bool = False) -> str:
    # Làm sạch text tiếng Việt cho TF-IDF + SVM.
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.lower().strip()

    # Map emoji/icon có sentiment thành token trước khi xóa ký tự đặc biệt.
    text = re.sub(POSITIVE_EMOJI_PATTERN, " emoji_positive ", text)
    text = re.sub(NEGATIVE_EMOJI_PATTERN, " emoji_negative ", text)

    # Xóa HTML, URL, email
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " url ", text)
    text = re.sub(r"\S+@\S+", " email ", text)

    # Giảm ký tự lặp dài
    text = reduce_repeated_chars(text, max_repeat=2)

    # Tách một số dấu câu thành khoảng trắng
    text = re.sub(r"[\n\r\t]+", " ", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ_\s]", " ", text)

    # Gộp khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    # Chuẩn hóa teencode sau khi đã lower và clean cơ bản
    text = normalize_teencode(text)

    if remove_stopwords:
        tokens = [tok for tok in text.split() if tok not in VI_STOPWORDS]
        text = " ".join(tokens)

    return text

# Test nhanh hàm clean
examples = [
    "Sản phẩm đẹpppppp 😍😍😍 shop giao hàng nhanhhhhh, rất okkkk!!!",
    "Hàng lỗi, giao chậm quá :((( huhu, ko hài lòng",
    "Sp dùng bt, đóng gói ổn, giá cũng được",
]

for ex in examples:
    print("RAW  :", ex)
    print("CLEAN:", clean_text(ex, remove_stopwords=False))
    print()

# 10. Optional — Word segmentation cho tiếng Việt

Tiếng Việt có nhiều từ ghép gồm nhiều âm tiết:

```text
sản phẩm
chất lượng
giao hàng
bình thường
```

Nếu chỉ split theo khoảng trắng, model có thể xem `sản` và `phẩm` là hai token riêng. Word segmentation sẽ nối lại:

```text
sản_phẩm
chất_lượng
giao_hàng
bình_thường
```

Điều này thường giúp model hiểu cụm từ tốt hơn.

Notebook này sẽ:

- dùng `underthesea` nếu bạn đã cài,
- nếu chưa cài thì tự bỏ qua và dùng text đã clean.

In [ ]:
USE_WORD_SEGMENTATION = True  # Đổi thành True nếu bạn đã cài underthesea và muốn dùng.

try:
    if USE_WORD_SEGMENTATION:
        from underthesea import word_tokenize
        print("Đã import underthesea thành công.")
    else:
        word_tokenize = None
        print("USE_WORD_SEGMENTATION=False, bỏ qua tách từ.")
except Exception as e:
    word_tokenize = None
    USE_WORD_SEGMENTATION = False
    print("Không import được underthesea, bỏ qua tách từ. Lỗi:", e)


def maybe_word_segment(text: str) -> str:
    if USE_WORD_SEGMENTATION and word_tokenize is not None:
        try:
            return word_tokenize(text, format="text")
        except Exception:
            return text
    return text

In [ ]:
# Apply preprocessing
USE_STOPWORDS = False  # Có thể thử True/False để so sánh.

data_clean = data.copy()
data_clean["text_clean"] = data_clean["text"].apply(lambda x: clean_text(x, remove_stopwords=USE_STOPWORDS))
data_clean["text_clean"] = data_clean["text_clean"].apply(maybe_word_segment)

# Loại bỏ text rỗng sau khi clean
data_clean = data_clean[data_clean["text_clean"].str.len() > 0].copy()

print("Shape sau khi clean:", data_clean.shape)
data_clean[["text", "text_clean", "label"]].head(10)

# 11. So sánh trước và sau preprocessing

Đây là bước giúp mình kiểm tra hàm clean có làm hỏng dữ liệu không.

Nếu thấy text sau clean mất quá nhiều thông tin, ta cần chỉnh lại function `clean_text()`.

In [ ]:
for idx in data_clean.sample(min(5, len(data_clean)), random_state=RANDOM_STATE).index:
    print("=" * 100)
    print("LABEL:", data_clean.loc[idx, "label"])
    print("RAW  :", data_clean.loc[idx, "text"])
    print("CLEAN:", data_clean.loc[idx, "text_clean"])

# 12. Train/Test split

Ta chia dữ liệu thành:

- **train set**: dùng để model học
- **test set**: dùng để đánh giá cuối cùng

Với classification, nên dùng `stratify=y` để giữ tỷ lệ class giữa train/test gần giống nhau.

In [ ]:
X = data_clean["text_clean"]
y = data_clean["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train size:", len(X_train))
print("Test size :", len(X_test))

split_summary = pd.DataFrame({
    "train_count": y_train.value_counts().sort_index(),
    "train_percent": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "test_count": y_test.value_counts().sort_index(),
    "test_percent": (y_test.value_counts(normalize=True).sort_index() * 100).round(2),
})
split_summary

# 13. Baseline model — DummyClassifier

Trước khi train model thật, ta tạo một baseline rất đơn giản.

Ví dụ: `DummyClassifier(strategy="most_frequent")` luôn dự đoán class xuất hiện nhiều nhất.

Nếu model SVM của ta không tốt hơn baseline nhiều, tức pipeline có vấn đề.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)

print("Dummy Accuracy:", accuracy_score(y_test, y_pred_dummy))
print("Dummy Macro F1 :", f1_score(y_test, y_pred_dummy, average="macro"))
print("\nClassification report:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

# 14. Model chính — TF-IDF + Linear SVM

Ta dùng `Pipeline` để đóng gói hai bước:

```text
TfidfVectorizer → LinearSVC
```

## Tham số TF-IDF quan trọng

```python
ngram_range=(1, 2)
```

Nghĩa là dùng:

- unigram: từng token riêng lẻ, ví dụ `đẹp`, `lỗi`
- bigram: cụm 2 token, ví dụ `giao hàng`, `không tốt`, `rất đẹp`

```python
min_df=2
```

Bỏ các token xuất hiện quá ít, giúp giảm noise.

```python
max_df=0.95
```

Bỏ các token xuất hiện trong quá nhiều document, vì thường ít phân biệt.

```python
sublinear_tf=True
```

Giảm ảnh hưởng của việc một từ lặp quá nhiều trong cùng một review.

## Tham số SVM quan trọng

```python
C=1.0
```

- C nhỏ: regularization mạnh hơn, model đơn giản hơn.
- C lớn: regularization yếu hơn, model cố fit train data kỹ hơn.

```python
class_weight="balanced"
```

Hữu ích nếu dataset bị lệch class.

In [ ]:
svm_pipeline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=50000,
        sublinear_tf=True,
    )),
    ("svm", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        max_iter=5000,
    ))
])

svm_pipeline.fit(X_train, y_train)
print("Train TF-IDF + LinearSVC xong!")

In [ ]:
y_pred = svm_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print("Kết quả TF-IDF + LinearSVC")
print("Accuracy       :", round(acc, 4))
print("Precision macro:", round(precision_macro, 4))
print("Recall macro   :", round(recall_macro, 4))
print("F1 macro       :", round(f1_macro, 4))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

# 15. Giải thích metrics trong project này

## Accuracy

Tỷ lệ dự đoán đúng trên toàn bộ test set.

$$
Accuracy = \frac{Số\ dự\ đoán\ đúng}{Tổng\ số\ mẫu}
$$

Accuracy dễ hiểu nhưng có thể gây hiểu nhầm nếu class bị lệch.

## Precision

Trong các mẫu model dự đoán là một class, có bao nhiêu mẫu thật sự đúng class đó.

Ví dụ với class `negative`:

```text
Model nói 100 review là negative, trong đó 80 review thật sự negative
Precision = 80 / 100 = 0.8
```

## Recall

Trong tất cả mẫu thật sự thuộc một class, model bắt được bao nhiêu.

Ví dụ với class `negative`:

```text
Có 100 review thật sự negative, model bắt đúng 70 review
Recall = 70 / 100 = 0.7
```

## F1-score

F1 là trung bình điều hòa giữa Precision và Recall.

$$
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
$$

Với sentiment analysis, mình thường nhìn:

- `macro F1`: công bằng giữa các class, phù hợp khi class imbalance.
- `weighted F1`: có tính đến số lượng mẫu mỗi class.

# 16. Confusion Matrix

Confusion Matrix cho biết model nhầm class nào với class nào.

Ví dụ nếu nhiều review `neutral` bị nhầm thành `positive`, có thể vì review trung tính chứa các từ như `ổn`, `được`, `bình thường`.

In [ ]:
labels_order = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels_order)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format="d")
plt.title("Confusion Matrix - TF-IDF + LinearSVC")
plt.show()

# 17. Cross-validation

Train/test split chỉ đánh giá một lần. Cross-validation giúp đánh giá ổn định hơn.

Ý tưởng:

```text
Chia train data thành k phần
Lần 1: train 4 phần, validate 1 phần
Lần 2: train 4 phần, validate 1 phần khác
...
Lấy trung bình kết quả
```

Ở đây ta dùng `StratifiedKFold` để giữ tỷ lệ class ở các fold.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "f1_macro": "f1_macro",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
}

cv_results = cross_validate(
    svm_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=False,
)

cv_summary = pd.DataFrame(cv_results).agg(["mean", "std"]).T
cv_summary

# 18. Hyperparameter tuning với GridSearchCV

Ta sẽ thử một vài cấu hình:

- `ngram_range`: dùng unigram hay unigram + bigram
- `min_df`: bỏ từ quá hiếm ở mức nào
- `C`: mức regularization của SVM

Metric chọn model: `f1_macro`

Vì sentiment dataset có thể lệch class, `f1_macro` thường đáng tin hơn accuracy.

> Nếu máy yếu, cell này có thể chạy hơi lâu. Bạn có thể giảm param grid hoặc bỏ qua lần đầu.

In [ ]:
RUN_GRID_SEARCH = True

param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__max_df": [0.9, 0.95],
    "svm__C": [0.5, 1.0, 2.0],
}

if RUN_GRID_SEARCH:
    grid_search = GridSearchCV(
        estimator=svm_pipeline,
        param_grid=param_grid,
        scoring="f1_macro",
        cv=3,
        n_jobs=-1,
        verbose=1,
    )

    grid_search.fit(X_train, y_train)

    print("Best params:")
    print(grid_search.best_params_)
    print("Best CV f1_macro:", round(grid_search.best_score_, 4))

    best_model = grid_search.best_estimator_
else:
    best_model = svm_pipeline
    print("Bỏ qua GridSearchCV, dùng svm_pipeline ban đầu.")

In [ ]:
# Đánh giá best model trên test set
best_pred = best_model.predict(X_test)

best_metrics = {
    "accuracy": accuracy_score(y_test, best_pred),
    "precision_macro": precision_score(y_test, best_pred, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, best_pred, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, best_pred, average="macro", zero_division=0),
}

pd.DataFrame([best_metrics]).round(4)

In [ ]:
print(classification_report(y_test, best_pred, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=labels_order)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels_order)
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, values_format="d")
plt.title("Confusion Matrix - Best Model")
plt.show()

# 19. Error Analysis — phân tích các câu model dự đoán sai

Đây là phần rất quan trọng trong project thực tế.

Không chỉ nhìn score, ta cần hỏi:

- Model hay nhầm class nào?
- Những review sai có đặc điểm gì?
- Có label noise không?
- Có review quá ngắn không?
- Có review chứa cả positive và negative không?

Ví dụ review khó:

```text
Giao hàng hơi lâu nhưng sản phẩm dùng ổn
```

Câu này có cả ý negative (`giao hàng lâu`) và positive (`sản phẩm ổn`). Nếu dataset chỉ có một label, model có thể nhầm.

In [ ]:
def get_svm_confidence(model, texts):
    # Lấy độ tự tin tương đối từ decision_function của LinearSVC.
    # Đây không phải probability, chỉ là margin score.
    try:
        scores = model.decision_function(texts)
        if scores.ndim == 1:
            return np.abs(scores)
        return np.max(scores, axis=1)
    except Exception:
        return np.full(len(texts), np.nan)

error_df = pd.DataFrame({
    "text_clean": X_test.values,
    "true_label": y_test.values,
    "pred_label": best_pred,
})

error_df["confidence_margin"] = get_svm_confidence(best_model, X_test)
error_df["is_correct"] = error_df["true_label"] == error_df["pred_label"]

wrong_df = error_df[~error_df["is_correct"]].copy()
wrong_df = wrong_df.sort_values("confidence_margin", ascending=False)

print("Số câu dự đoán sai:", len(wrong_df))
print("Tỉ lệ sai (%):", round(len(wrong_df) / len(error_df) * 100, 2))

wrong_df.head(20)

In [ ]:
# Lưu wrong predictions để xem kỹ bên ngoài.
wrong_path = REPORT_DIR / "wrong_predictions.csv"
wrong_df.to_csv(wrong_path, index=False, encoding="utf-8-sig")
print("Đã lưu wrong predictions tại:", wrong_path)

In [ ]:
# Xem một số lỗi cụ thể
for i, row in wrong_df.head(10).iterrows():
    print("=" * 100)
    print("TRUE:", row["true_label"], "| PRED:", row["pred_label"], "| MARGIN:", round(row["confidence_margin"], 4))
    print(row["text_clean"])

# 20. Feature importance — từ nào ảnh hưởng mạnh đến từng class?

Với LinearSVC, model học một trọng số cho mỗi feature TF-IDF.

Feature có weight lớn với class nào thì thường là dấu hiệu mạnh cho class đó.

Ví dụ kỳ vọng:

- Positive: `đẹp`, `tốt`, `ưng`, `nhanh`, `chất lượng`, `cảm ơn`
- Negative: `lỗi`, `chậm`, `tệ`, `không hài lòng`, `thất vọng`, `bể`

Lưu ý: đây chỉ là diễn giải tương đối. Không phải lúc nào top feature cũng hoàn hảo.

In [ ]:
def show_top_features_per_class(model, top_n=20):
    tfidf = model.named_steps["tfidf"]
    svm = model.named_steps["svm"]

    feature_names = np.array(tfidf.get_feature_names_out())
    classes = svm.classes_
    coef = svm.coef_

    # Binary LinearSVC có coef_ shape (1, n_features)
    if coef.shape[0] == 1 and len(classes) == 2:
        weights = coef[0]
        print(f"Class negative-ish: {classes[0]}")
        top_negative = np.argsort(weights)[:top_n]
        print(feature_names[top_negative][::-1])

        print(f"\nClass positive-ish: {classes[1]}")
        top_positive = np.argsort(weights)[-top_n:]
        print(feature_names[top_positive][::-1])
        return

    for class_idx, class_name in enumerate(classes):
        print("=" * 100)
        print("CLASS:", class_name)
        top_idx = np.argsort(coef[class_idx])[-top_n:][::-1]
        top_features = pd.DataFrame({
            "feature": feature_names[top_idx],
            "weight": coef[class_idx][top_idx]
        })
        display(top_features)

show_top_features_per_class(best_model, top_n=20)

# 21. Predict review mới

Sau khi train xong, ta cần viết một function nhận câu review mới và trả về sentiment.

Pipeline khi predict:

```text
raw review
→ clean_text
→ optional word segmentation
→ best_model.predict
→ output label
```

In [ ]:
def preprocess_for_prediction(text: str) -> str:
    text = clean_text(text, remove_stopwords=USE_STOPWORDS)
    text = maybe_word_segment(text)
    return text


def predict_sentiment(text: str, model=best_model, verbose=True):
    text_clean = preprocess_for_prediction(text)
    pred = model.predict([text_clean])[0]

    # Lấy margin score nếu có
    confidence = get_svm_confidence(model, [text_clean])[0]

    result = {
        "raw_text": text,
        "clean_text": text_clean,
        "prediction": pred,
        "confidence_margin": float(confidence) if not pd.isna(confidence) else None,
    }

    if verbose:
        print("Raw text          :", result["raw_text"])
        print("Clean text        :", result["clean_text"])
        print("Prediction        :", result["prediction"])
        print("Confidence margin :", result["confidence_margin"])

    return result

In [ ]:
test_reviews = [
    "Sản phẩm rất đẹp, giao hàng nhanh, đóng gói cẩn thận, mình rất ưng 😍",
    "Hàng bị lỗi, chất lượng quá tệ, shop xử lý chậm, rất thất vọng",
    "Sản phẩm bình thường, dùng tạm được, không quá tốt cũng không quá tệ",
    "Shop giao thiếu hàng nhưng đóng gói cũng ổn",
]

for review in test_reviews:
    print("=" * 100)
    predict_sentiment(review)

# 22. Lưu model

Sau khi chọn được model tốt nhất, ta lưu lại bằng `joblib`.

File lưu gồm:

- pipeline TF-IDF + SVM,
- cấu hình preprocessing,
- danh sách label,
- thông tin dataset và metrics.

Sau này khi làm API hoặc Streamlit app, ta chỉ cần load file `.joblib` này để predict.

In [ ]:
model_package = {
    "model": best_model,
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "labels": sorted(y.unique()),
    "use_stopwords": USE_STOPWORDS,
    "use_word_segmentation": USE_WORD_SEGMENTATION,
    "metrics": best_metrics,
    # "selected_csv": str(SELECTED_CSV),
}

model_path = MODEL_DIR / "sentiment_tfidf_svm.joblib"
joblib.dump(model_package, model_path)

print("Đã lưu model tại:", model_path)

In [ ]:
# Test load lại model
loaded_package = joblib.load(model_path)
loaded_model = loaded_package["model"]

sample = "san pham qua te"
print("Predict sau khi load model:")
predict_sentiment(sample, model=loaded_model)

# 23. Tạo bảng kết quả tổng kết

Bảng này giúp bạn copy vào report hoặc README.

In [ ]:
summary_table = pd.DataFrame([
    {
        "model": "Dummy most_frequent",
        "accuracy": accuracy_score(y_test, y_pred_dummy),
        "precision_macro": precision_score(y_test, y_pred_dummy, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred_dummy, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred_dummy, average="macro", zero_division=0),
    },
    {
        "model": "TF-IDF + LinearSVC initial",
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
    },
    {
        "model": "TF-IDF + LinearSVC best",
        "accuracy": best_metrics["accuracy"],
        "precision_macro": best_metrics["precision_macro"],
        "recall_macro": best_metrics["recall_macro"],
        "f1_macro": best_metrics["f1_macro"],
    },
])

summary_table.round(4)

In [ ]:
summary_path = REPORT_DIR / "model_summary.csv"
summary_table.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("Đã lưu bảng summary tại:", summary_path)

# 24. Kết luận project

## Những gì đã làm được

Trong project này, ta đã hoàn thành một pipeline NLP Machine Learning đầy đủ:

```text
Load Shopee Vietnamese reviews
→ kiểm tra dữ liệu
→ chuẩn hóa label
→ EDA
→ clean text tiếng Việt
→ optional word segmentation
→ train/test split
→ baseline DummyClassifier
→ TF-IDF + LinearSVC
→ evaluate bằng Accuracy / Precision / Recall / F1
→ Confusion Matrix
→ Cross-validation
→ GridSearchCV tuning
→ Error Analysis
→ xem top features
→ predict review mới
→ save model bằng joblib
```

## Vì sao project này quan trọng?

Project này giúp bạn luyện đúng các kỹ năng NLP classic:

- **Text preprocessing**: làm sạch dữ liệu text thật.
- **Vectorization**: biến câu thành vector bằng TF-IDF.
- **Text classification**: phân loại review positive/negative/neutral.
- **SVM**: dùng LinearSVC cho dữ liệu sparse high-dimensional.
- **Evaluation**: không chỉ nhìn accuracy mà xem precision, recall, F1 và confusion matrix.
- **Error analysis**: đọc lại các câu model sai để cải thiện pipeline.

## Hạn chế

TF-IDF + SVM là baseline mạnh nhưng vẫn có hạn chế:

1. Không hiểu ngữ cảnh sâu như Transformer.
2. Khó xử lý câu mỉa mai hoặc câu có cảm xúc pha trộn.
3. Phụ thuộc nhiều vào preprocessing.
4. Nếu có từ mới hoặc teencode lạ, model có thể không hiểu tốt.

## Hướng nâng cấp

Sau khi nắm chắc project này, có thể nâng cấp theo các hướng:

1. So sánh với `LogisticRegression` và `MultinomialNB`.
2. Thử `char n-gram TF-IDF` để xử lý lỗi chính tả và teencode tốt hơn.
3. Dùng `underthesea` hoặc `pyvi` word segmentation nghiêm túc hơn.
4. Dùng PhoBERT fine-tuning khi chuyển sang Deep Learning/NLP nâng cao.
5. Build API bằng FastAPI:

```text
POST /predict
Input : {"review": "Sản phẩm rất tốt"}
Output: {"sentiment": "positive"}
```

# 25. Bài tập tự luyện thêm

Bạn có thể tự làm các task sau để hiểu project sâu hơn. Không cần làm hết một lần.

## Task 1 — So sánh stopwords

Chạy 2 lần:

```python
USE_STOPWORDS = False
USE_STOPWORDS = True
```

So sánh `f1_macro` thay đổi thế nào.

## Task 2 — So sánh n-gram

Thử các cấu hình:

```python
ngram_range=(1, 1)
ngram_range=(1, 2)
ngram_range=(1, 3)
```

Ghi lại model nào tốt hơn.

## Task 3 — Char n-gram

Thử thêm model:

```python
TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
```

Char n-gram đôi khi rất mạnh với tiếng Việt social/e-commerce vì xử lý được sai chính tả nhẹ.

## Task 4 — Error analysis sâu hơn

Mở file:

```text
reports/wrong_predictions.csv
```

Đọc 30 dòng sai và phân loại nguyên nhân sai:

- label noise,
- câu quá ngắn,
- câu có nhiều cảm xúc,
- teencode,
- sarcasm/mỉa mai,
- preprocessing làm mất thông tin.

## Task 5 — Làm mini API

Sau khi model ổn, tạo file `app.py` bằng FastAPI để predict sentiment từ review mới.